<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path.cwd()
raw_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
for parent in [repo_root, *repo_root.parents]:
    candidate = parent / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        raw_path = candidate
        repo_root = parent
        break

df = pd.read_csv(raw_path)

# Match the repository's feature-prep behavior closely.
for col in [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "trend_pct",
]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

for col in [
    "competition_level", "content_type", "main_intent", "provider_used",
    "model_used", "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "impression_tier", "position_tier", "trend_direction",
]:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

def reason_codes(row: pd.Series) -> list[str]:
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and ((row["engagement_rate"] > 0 and row["engagement_rate"] < 30) or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons

print(f"Prepared rows: {len(df):,}")
print("Example reason codes:", reason_codes(df.iloc[0]))

Prepared rows: 30,000
Example reason codes: ['declining_with_demand']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Reuse the same prepared frame from the previous cell.
# The repository's scoring script uses percentile ranks and a weighted composite.

def normalize(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    minimum = values.min()
    maximum = values.max()
    if not np.isfinite(minimum) or not np.isfinite(maximum) or maximum == minimum:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - minimum) / (maximum - minimum)


def percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)

# Build the baseline score.
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]

df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

df["reason_codes"] = df.apply(lambda row: "|".join(reason_codes(row)), axis=1)

def suggested_action(row: pd.Series) -> str:
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

# Rank and keep a clear output frame.
df["suggested_action_baseline"] = df.apply(suggested_action, axis=1)
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]

queue = df[output_columns].sort_values("baseline_rank").reset_index(drop=True)
queue_path = Path("work/outputs/baseline_action_score.csv")
queue_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(queue_path, index=False)

print(f"Wrote {queue_path}")
print(queue.head(10)[["baseline_rank", "baseline_refresh_score", "reason_codes", "suggested_action_baseline"]])

Wrote work\outputs\baseline_action_score.csv
   baseline_rank  baseline_refresh_score  \
0              1                0.941189   
1              2                0.934889   
2              3                0.934080   
3              4                0.933606   
4              5                0.933559   
5              6                0.933263   
6              7                0.932991   
7              8                0.931623   
8              9                0.931363   
9             10                0.931124   

                                        reason_codes suggested_action_baseline  
0  declining_with_demand|page_one_decay_risk|low_...                   refresh  
1    page_one_decay_risk|low_engagement_visible_page                   monitor  
2    page_one_decay_risk|low_engagement_visible_page                   monitor  
3  page_one_decay_risk|low_ctr_visible_page|low_e...    refresh_and_review_ctr  
4  page_one_decay_risk|low_ctr_visible_page|low_e...    refresh_a

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Review the top 20 items from the ranked queue.
review = queue.head(20).copy()
review[["content_id", "baseline_rank", "baseline_refresh_score", "reason_codes", "suggested_action_baseline", "is_declining_label", "avg_position", "ctr", "days_since_last_update"]].head(20)

,content_id,baseline_rank,baseline_refresh_score,reason_codes,suggested_action_baseline,is_declining_label,avg_position,ctr,days_since_last_update
0,content_9532f197bbc8,1,0.941189,declining_with_demand|page_one_decay_risk|low_...,refresh,1,2.0,0.87,104
1,content_4d1fe5b32dc2,2,0.934889,page_one_decay_risk|low_engagement_visible_page,monitor,0,2.5,0.52,104
2,content_07f2e7a6f38a,3,0.934080,page_one_decay_risk|low_engagement_visible_page,monitor,0,2.7,0.85,104
3,content_e5ae436f9a16,4,0.933606,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,3.0,0.45,104
4,content_3430a8b94511,5,0.933559,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,3.3,0.29,104
5,content_cbd93118300b,6,0.933263,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,1,3.3,0.46,104
6,content_9c195417f6ef,7,0.932991,page_one_decay_risk|low_engagement_visible_page,monitor,0,2.5,0.73,104
7,content_ba2acb4ebd04,8,0.931623,page_one_decay_risk|low_engagement_visible_page,monitor,0,3.6,0.83,104
8,content_79b25654070a,9,0.931363,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0,3.7,0.48,104
9,content_adddad39251c,10,0.931124,page_one_decay_risk|low_engagement_visible_page,monitor,0,3.6,0.55,104


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Leakage sanity check: confirm no label-source or future-window feature is included in the score.
leakage_checks = {
    "label_source_included": "trend_pct" in queue.columns,
    "target_column_present": "is_declining_label" in queue.columns,
    "future_window_features_present": any(col in df.columns for col in ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]),
}

# A simple note for the markdown section below.
leakage_checks

{'label_source_included': False,
 'target_column_present': True,
 'future_window_features_present': True}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.